In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

# 1. Configuration for your specific target run
target_config = {
    'Depth': 3,
    'Filters': 64,
    'LR': 0.0005
}

# 2. Load and Filter Data
df = pd.read_csv("training_logs/grid_search_*.csv")
# Assuming 'df' is the concatenated dataframe from your previous blocks
# Filtering for your specific best-performing architecture
best_model_df = df[
    (df['Depth'] == target_config['Depth']) & 
    (df['Filters'] == target_config['Filters']) & 
    (df['LR'] == target_config['LR'])
]

# 3. Calculate 5-Fold Cross Validation Statistics
# Grouping by Fold and taking the best (minimum) error or final epoch values
cv_metrics = best_model_df.groupby('Fold').agg({
    'Val_MSE': 'min',
    'Val_RMSE': 'min',
    'Val_MAE': 'min',
    'Val_R2': 'max'
})

print(f"--- 5-Fold Summary for Depth {target_config['Depth']}, {target_config['Filters']} Filters ---")
stats_summary = pd.DataFrame({
    'Mean': cv_metrics.mean(),
    'SD': cv_metrics.std()
})
print(stats_summary)

# 4. Create the Visualization Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Convergence of all 5 Folds (Test Error)
sns.lineplot(ax=axes[0], data=best_model_df, x='Epoch', y='Val_MSE', hue='Fold', palette='viridis')
axes[0].set_title(f"Test MSE Convergence across 5 Folds\n(Depth {target_config['Depth']}, {target_config['Filters']} Filters)")
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot B: Performance Metrics Comparison (Bar Chart)
metrics_to_plot = cv_metrics.mean().reset_index()
metrics_to_plot.columns = ['Metric', 'Value']
sns.barplot(ax=axes[1], data=metrics_to_plot, x='Metric', y='Value', palette='Set2')
axes[1].set_title("Average Performance Across All Metrics")
# Add error bars based on SD
for i, metric in enumerate(metrics_to_plot['Metric']):
    axes[1].errorbar(i, metrics_to_plot['Value'][i], yerr=stats_summary.loc[metric, 'SD'], fmt='none', c='black', capsize=5)

plt.tight_layout()
plt.show()